In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
DATASET_DIR = "/kaggle/input/datasets/aviarvasu/ds1-split"          
TRAIN_DIR   = os.path.join(/kaggle/input/datasets/aviarvasu/ds1-split/DS1_split/TRAIN "TRAIN")
TEST_DIR    = os.path.join(/kaggle/input/datasets/aviarvasu/ds1-split/DS1_split/TEST, "TEST")
WORK_DIR    = "/kaggle/working"

In [ ]:
IMG_SIZE      = (224, 224)
BATCH_SIZE    = 1024                     
EPOCHS_FROZEN = 15
EPOCHS_FINE   = 20
LR_FROZEN     = 1e-3 * (BATCH_SIZE / 32) 
LR_FINE       = 1e-5 * (BATCH_SIZE / 32)



In [ ]:
CLASS_NAMES   = ["N", "O", "R"]
DISPLAY_NAMES = ["Non-Recyclable", "Organic", "Recyclable"]
NUM_CLASSES   = len(CLASS_NAMES)

MODEL_SAVE_PATH = os.path.join(WORK_DIR, "waste_densenet121_final.keras")

print(f"\nTrain dir : {TRAIN_DIR}")
print(f"Test  dir : {TEST_DIR}")
print(f"Batch size: {BATCH_SIZE}  |  LR frozen: {LR_FROZEN:.5f}  |  LR fine: {LR_FINE:.7f}")

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def get_file_paths_and_labels(root, classes):
    paths, labels = [], []
    for idx, cls in enumerate(classes):
        cls_dir = os.path.join(root, cls)
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
                paths.append(os.path.join(cls_dir, fname))
                labels.append(idx)
    return paths, labels

def parse_image(path, label, augment=False):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)          
    img = tf.image.resize(img, IMG_SIZE)
    img = preprocess_input(img)                           

    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.2)
        img = tf.image.random_contrast(img, 0.8, 1.2)
        img = tf.image.random_saturation(img, 0.8, 1.2)

        # Random rotation ±20° via tfa-free approach
        angle = tf.random.uniform([], -0.35, 0.35)       
        img = _rotate(img, angle)

        # Random zoom via crop-and-resize
        img = _random_zoom(img, zoom_range=0.20)

    label_oh = tf.one_hot(label, NUM_CLASSES)
    return img, label_oh

def _rotate(img, angle):
    """Rotate image by `angle` radians using affine transform."""
    img = tf.expand_dims(img, 0)
    cos_a, sin_a = tf.math.cos(angle), tf.math.sin(angle)
    transform = [cos_a, -sin_a, 0.0, sin_a, cos_a, 0.0, 0.0, 0.0]
    img = tf.raw_ops.ImageProjectiveTransformV3(
        images=img,
        transforms=[transform],
        output_shape=IMG_SIZE,
        interpolation="BILINEAR",
        fill_mode="REFLECT",
    )
    return tf.squeeze(img, 0)

def _random_zoom(img, zoom_range=0.20):
    scale = tf.random.uniform([], 1.0 - zoom_range, 1.0 + zoom_range)
    h, w  = IMG_SIZE
    new_h = tf.cast(tf.cast(h, tf.float32) / scale, tf.int32)
    new_w = tf.cast(tf.cast(w, tf.float32) / scale, tf.int32)
    new_h = tf.clip_by_value(new_h, 1, h)
    new_w = tf.clip_by_value(new_w, 1, w)
    img   = tf.image.resize_with_crop_or_pad(img, new_h, new_w)
    img   = tf.image.resize(img, IMG_SIZE)
    return img

def build_dataset(root, classes, batch_size, augment=False, repeat=False):
    paths, labels = get_file_paths_and_labels(root, classes)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if augment:
        ds = ds.shuffle(buffer_size=len(paths), seed=42)
    ds = ds.map(
        lambda p, l: parse_image(p, l, augment=augment),
        num_parallel_calls=AUTOTUNE,
    )
    ds = ds.batch(batch_size, drop_remainder=True)   
    if repeat:
        ds = ds.repeat()
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = build_dataset(TRAIN_DIR, CLASS_NAMES, BATCH_SIZE, augment=True,  repeat=True)
test_ds  = build_dataset(TEST_DIR,  CLASS_NAMES, BATCH_SIZE, augment=False, repeat=False)

STEPS_PER_EPOCH       = NUM_TRAIN // BATCH_SIZE
VALIDATION_STEPS      = NUM_TEST  // BATCH_SIZE

print(f"\nSteps/epoch (train): {STEPS_PER_EPOCH}")
print(f"Validation steps   : {VALIDATION_STEPS}")

In [ ]:
with strategy.scope():
    base_model = DenseNet121(
        weights="imagenet",
        include_top=False,
        input_shape=(*IMG_SIZE, 3),
    )
    base_model.trainable = False  

    inputs  = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x       = base_model(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(256, activation="relu")(x)
    x       = layers.Dropout(0.4)(x)
    x       = layers.Dense(128, activation="relu")(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(inputs, outputs, name="WasteClassifier_DenseNet121")

    model.compile(
        optimizer=optimizers.Adam(learning_rate=LR_FROZEN),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

print(f"\nTotal params    : {model.count_params():,}")
model.summary()

In [ ]:
# PHASE 1: Training Head Only
cb_phase1 = [
    callbacks.EarlyStopping(monitor="val_accuracy", patience=4,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                patience=3, min_lr=1e-7, verbose=1),
    callbacks.ModelCheckpoint(
        os.path.join(WORK_DIR, "best_phase1_densenet.keras"),
        monitor="val_accuracy", save_best_only=True, verbose=1,
    ),
]

print("\nPhase 1 — Training head only (base frozen)")
history1 = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS_FROZEN,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=VALIDATION_STEPS,
    callbacks=cb_phase1,
    class_weight=class_weight_dict,
)

In [ ]:
# PHASE 2: Fine-tuning of top 30 Layers
with strategy.scope():
    base_model.trainable = True
    for layer in base_model.layers[:-30]:
        layer.trainable = False

    model.compile(
        optimizer=optimizers.Adam(learning_rate=LR_FINE),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

cb_phase2 = [
    callbacks.EarlyStopping(monitor="val_accuracy", patience=5,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                patience=3, min_lr=1e-8, verbose=1),
    callbacks.ModelCheckpoint(
        os.path.join(WORK_DIR, "best_phase2_densenet.keras"),
        monitor="val_accuracy", save_best_only=True, verbose=1,
    ),
]

train_ds = build_dataset(TRAIN_DIR, CLASS_NAMES, BATCH_SIZE, augment=True,  repeat=True)
test_ds  = build_dataset(TEST_DIR,  CLASS_NAMES, BATCH_SIZE, augment=False, repeat=False)

print("\nPhase 2 — Fine-tuning top 30 layers of DenseNet121 (last dense block)")
history2 = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS_FINE,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=VALIDATION_STEPS,
    callbacks=cb_phase2,
    class_weight=class_weight_dict,
)


In [ ]:
#plotting training history
def plot_history(h1, h2, metric, ax):
    full_train = h1.history[metric]        + h2.history[metric]
    full_val   = h1.history[f"val_{metric}"] + h2.history[f"val_{metric}"]
    split      = len(h1.history[metric])
    ax.plot(full_train, label=f"Train {metric}")
    ax.plot(full_val,   label=f"Val {metric}")
    ax.axvline(x=split - 1, color="gray", linestyle="--", label="Phase 2 start")
    ax.set_xlabel("Epoch"); ax.set_ylabel(metric.capitalize())
    ax.set_title(f"Training & Validation {metric.capitalize()}"); ax.legend()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
plot_history(history1, history2, "accuracy", ax1)
plot_history(history1, history2, "loss",     ax2)
plt.suptitle("DenseNet121 — Training History (Phase 1 + Phase 2)", fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
#evaluation
print("\nEvaluating on Test Data...")
test_ds_eval = build_dataset(TEST_DIR, CLASS_NAMES, BATCH_SIZE, augment=False, repeat=False)
test_loss, test_accuracy = model.evaluate(test_ds_eval, steps=VALIDATION_STEPS)
print(f"Final Test Accuracy: {test_accuracy * 100:.2f}%")


print("Generating predictions...")
test_ds_pred = build_dataset(TEST_DIR, CLASS_NAMES, BATCH_SIZE, augment=False, repeat=False)
predictions      = model.predict(test_ds_pred, steps=VALIDATION_STEPS)
predicted_labels = np.argmax(predictions, axis=1)


_, true_label_list = get_file_paths_and_labels(TEST_DIR, CLASS_NAMES)
true_labels = np.array(true_label_list[:len(predicted_labels)])

print("\n--- Classification Report ---")
print(classification_report(true_labels, predicted_labels, target_names=DISPLAY_NAMES))

print("\n--- Confusion Matrix ---")
cm = confusion_matrix(true_labels, predicted_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=DISPLAY_NAMES, yticklabels=DISPLAY_NAMES)
plt.title("Waste Classification — Confusion Matrix (DenseNet121)")
plt.ylabel("Actual True Label"); plt.xlabel("Model Prediction")
plt.tight_layout(); plt.show()


In [ ]:
print(f"\nSaving to: {MODEL_SAVE_PATH}")
model.save(MODEL_SAVE_PATH)
print(f"Model saved")